In [1]:
from transformers import AutoTokenizer
from pathlib import Path

NB_DIR = Path().resolve()
ROOT_DIR = (NB_DIR / ".." / "..").resolve()
MODEL_ID = Path(ROOT_DIR) / "modelos" / "deepseek-coder-1p3b-lora"

ft_dir = Path(ROOT_DIR) / "modelos" / "deepseek-coder-1p3b-lora"
merged_dir = Path(ROOT_DIR) / "modelos" / "deepseek-coder-1p3b-merged"


In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch
from pathlib import Path

# 1. Rutas
BASE_MODEL_ID = "deepseek-ai/deepseek-coder-1.3b-base"
LORA_ADAPTER_PATH = Path(ROOT_DIR) / "modelos" / "deepseek-coder-1p3b-lora" # cambia a tu ruta real
MERGED_OUTPUT_PATH = Path(ROOT_DIR) / "modelos" / "deepseek-coder-1p3b-merged"

Path(MERGED_OUTPUT_PATH).mkdir(parents=True, exist_ok=True)

# 2. Cargar tokenizer (desde el modelo base o tu checkpoint)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)

# 3. Cargar modelo base
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16,    # o bfloat16 / float32 según tu hardware
    device_map="auto",            # o "cpu" si no tienes GPU
    trust_remote_code=True,
)

# 4. Cargar adaptador LoRA entrenado
model = PeftModel.from_pretrained(
    model,
    LORA_ADAPTER_PATH,
    torch_dtype=torch.float16,
)

# 5. Hacer el merge de LoRA en el modelo base
#    (después de esto, el modelo deja de ser PEFT y se vuelve "normal")
model = model.merge_and_unload()  # <- esta es la magia

# 6. Guardar el modelo fusionado
model.save_pretrained(MERGED_OUTPUT_PATH)
tokenizer.save_pretrained(MERGED_OUTPUT_PATH)

print(f"Modelo fusionado guardado en: {MERGED_OUTPUT_PATH}")


Skipping import of cpp extensions due to incompatible torch version 2.6.0+cu124 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info
`torch_dtype` is deprecated! Use `dtype` instead!


Modelo fusionado guardado en: /home/jsoa/competencia_kaggle/Proyecto-PLN-FLAG/modelos/deepseek-coder-1p3b-merged


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer

LOCAL_MODEL_PATH = Path(ROOT_DIR) / "modelos" / "deepseek-coder-1p3b-merged"


tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_PATH, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    LOCAL_MODEL_PATH,
    torch_dtype="auto",
    device_map="auto",  # o 'cpu'
    trust_remote_code=True,
)

prompt = "Escribe una función en Python que calcule el factorial de un número."
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=200)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


`torch_dtype` is deprecated! Use `dtype` instead!
Some parameters are on the meta device because they were offloaded to the cpu.
Setting `pad_token_id` to `eos_token_id`:32014 for open-end generation.


Escribe una función en Python que calcule el factorial de un n�mero.

def factorial(n):
    if n == 0:
        return 1
    else:
        return n * factorial(n-1)

print(factorial(5))

# 5 * 4 * 3 * 2 * 1

# 5 * 4 * 3 * 2

# 5 * 4 * 3

# 5 * 4

# 5 * 3

# 5 * 2

# 5 * 1

# 5

# 120

# 5 * 4 * 3 * 2 * 1

# 5 * 4 * 3 * 2

# 5 * 4 * 3

# 5 * 4

# 5 * 3

# 5 * 2

#
